---

Reconecte ao Drive.

In [1]:
import os
DRIVE_DIRECTORY = "curso_ml"
DRIVE_DIRECTORY = os.path.join("/content/drive/MyDrive", DRIVE_DIRECTORY)

---

In [2]:
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## K-means

### Base de dados cobertura vegetal

Neste exercício você vai usar a base de dados de cobertura vegetal, explorada nos exercícios da **Parte 1 - Classificação**.

Comece carregando os dados do arquivo `cov_types.csv` armazenado na pasta do Drive.

In [3]:
base = pd.read_csv(os.path.join(DRIVE_DIRECTORY, 'cov_types.csv'))

Exiba os dados para uma inspeção inicial.

In [4]:
base

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type,Cover_Type
0,2767.0,66.0,17.0,210.0,18.0,1190.0,234.0,204.0,96.0,2251.0,2,30,Lodgepole Pine
1,2724.0,160.0,19.0,60.0,4.0,1350.0,236.0,240.0,127.0,2514.0,2,16,Lodgepole Pine
2,2360.0,65.0,7.0,127.0,21.0,1377.0,227.0,226.0,134.0,339.0,3,5,Ponderosa Pine
3,2995.0,45.0,4.0,285.0,30.0,5125.0,221.0,231.0,146.0,5706.0,0,11,Lodgepole Pine
4,2400.0,106.0,27.0,150.0,63.0,342.0,253.0,196.0,51.0,811.0,2,3,Ponderosa Pine
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2917.0,90.0,9.0,247.0,25.0,4095.0,235.0,225.0,121.0,3901.0,0,28,Lodgepole Pine
9996,3015.0,38.0,8.0,361.0,74.0,4846.0,220.0,223.0,138.0,1611.0,0,28,Lodgepole Pine
9997,3052.0,79.0,19.0,90.0,11.0,1003.0,241.0,203.0,85.0,1490.0,2,22,Spruce/Fir
9998,2958.0,58.0,6.0,319.0,19.0,2468.0,225.0,227.0,137.0,2280.0,0,28,Lodgepole Pine


Desta vez, nós vamos remover os atributos categóricos, `Wilderness_Area` e `Soil_Type`, pois para apresentá-los ao algoritmo, nós precisaríamos usar One Hot Encoding, o que adiciona muitas colunas no dataset, e como nossos dados são limitados a 10 mil exemplos, isso pode prejudicar o algoritmo K-Means.

In [5]:
base = base.drop(["Wilderness_Area", "Soil_Type"], axis=1)
base

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type
0,2767.0,66.0,17.0,210.0,18.0,1190.0,234.0,204.0,96.0,2251.0,Lodgepole Pine
1,2724.0,160.0,19.0,60.0,4.0,1350.0,236.0,240.0,127.0,2514.0,Lodgepole Pine
2,2360.0,65.0,7.0,127.0,21.0,1377.0,227.0,226.0,134.0,339.0,Ponderosa Pine
3,2995.0,45.0,4.0,285.0,30.0,5125.0,221.0,231.0,146.0,5706.0,Lodgepole Pine
4,2400.0,106.0,27.0,150.0,63.0,342.0,253.0,196.0,51.0,811.0,Ponderosa Pine
...,...,...,...,...,...,...,...,...,...,...,...
9995,2917.0,90.0,9.0,247.0,25.0,4095.0,235.0,225.0,121.0,3901.0,Lodgepole Pine
9996,3015.0,38.0,8.0,361.0,74.0,4846.0,220.0,223.0,138.0,1611.0,Lodgepole Pine
9997,3052.0,79.0,19.0,90.0,11.0,1003.0,241.0,203.0,85.0,1490.0,Spruce/Fir
9998,2958.0,58.0,6.0,319.0,19.0,2468.0,225.0,227.0,137.0,2280.0,Lodgepole Pine


Separe a coluna `Cover_Type` em uma variável `y` (já que esta é a classe do dataset), e utilize o método `value_counts` para relembrar quantas classes esse problema contém, e qual sua frequência.

In [6]:
y = base['Cover_Type']
y.value_counts()

Cover_Type
Lodgepole Pine       4847
Spruce/Fir           3714
Ponderosa Pine        581
Krummholz             362
Douglas-fir           278
Aspen                 163
Cottonwood/Willow      55
Name: count, dtype: int64

Agora separe o restante das colunas em uma variável `X`, e já recupere o atributo `values` para ter um NumPy array. Exiba este array.

In [7]:
X = base.iloc[:,1].values
X

array([ 66., 160.,  65., ...,  79.,  58.,  91.])

Como os dados estão em escalas diferentes, instancie um escalonador do tipo `StandardScaler` e escalone `X` com ele.

In [10]:
scaler = StandardScaler()
X = X.reshape(-1, 1)
X = scaler.fit_transform(X)
X

array([[-0.78934465],
       [ 0.05516249],
       [-0.79832877],
       ...,
       [-0.67255111],
       [-0.8612176 ],
       [-0.56474169]])

Agora instancie e ajuste um algoritmo do tipo `KMeans`. Utilize `n_clusters` igual ao número de classes do dataset.

In [11]:
km = KMeans(n_clusters=7)
km.fit(X)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",7
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",None
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


Recupere e exiba os centroides.

In [12]:
centroids = km.cluster_centers_
centroids

array([[ 1.62268527],
       [-0.40996459],
       [ 0.55373864],
       [-1.19404867],
       [ 1.12838411],
       [ 0.04532821],
       [-0.81816949]])

Aplique o método `inverse_transform` do escalonador para visualizar os centroides na escala original.

In [13]:
scaler.inverse_transform(centroids)

array([[334.47708199],
       [108.22785623],
       [215.49527807],
       [ 20.95339697],
       [279.45764273],
       [158.90537084],
       [ 62.79157895]])

Recupere e exiba os rótulos atribuídos pelo algoritmo KMeans em uma variável `labels`.

In [14]:
labels = km.labels_
labels

array([6, 5, 6, ..., 6, 6, 1])

Agora aplique o método do cotovelo para verificar se o número de clusters igual a 7 é o melhor valor para agrupar este dataset. Faça uma pesquisa com 1 até 10 clusters, e plote o gráfico correspondente.

In [ ]:
___

In [ ]:
fig = px.line(x=range(___), ___)
fig.show()

Qual sua conclusão a partir do gráfico?

Vamos visualizar os dados junto com os rótulos atribuídos. Para isso, precisamos utilizar a técnica PCA.

Instancie um objeto PCA com `n_components=2`, e transforme `X` para seu PCA correspondente.

In [ ]:
pca = ___
___

Agora utilize `X_pca` para plotar os dados, usando `labels` para definir a cor dos pontos. Desta vez, utilize `labels.astype(str)` para que a biblioteca considere que os rótulos são categóricos, e não numéricos.

In [ ]:
fig = px.___(x=___, y=___, color=___.as___(___))
fig.show()

Para comparar, refaça o mesmo gráfico, mas agora utilize `y` (ou seja, os rótulos originais) para definir a cor dos pontos.

In [ ]:
fig = px.scatter(x=___, y=___, color=___)
fig.show()

Observe que a comparação deixa evidente que o agrupamento baseado apenas nos atributos resulta em grupos bem diferentes daqueles discriminados pelas espécies das árvores.

DICA: você pode comparar as classes originais com os rótulos gerados pelo algoritmo utilizando a função `pd.crosstab`:

In [ ]:
pd.crosstab(labels, y)

Isso indica que nenhum rótulo é característico de apenas uma classe, e vice-versa, o que confirma a visualização do gráfico.